# Assignment 2 — CartPole (CUDA, DQN and PPO)

Run top to bottom; save this notebook **with outputs** before submission.

## 1. CUDA setup and reproducibility

Choose the NVIDIA CUDA kernel. This run writes only to the new output folder.

In [ ]:
from pathlib import Path
import csv, json, random, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gymnasium as gym
import torch as th
import torch.nn as nn
from IPython.display import display, Video
from stable_baselines3 import DQN, PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.vec_env import DummyVecEnv

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
th.manual_seed(SEED)
if not th.cuda.is_available():
    raise RuntimeError("Select the NVIDIA CUDA Python kernel before running training.")
DEVICE = "cuda:0"
print("Python:", sys.executable)
print("PyTorch:", th.__version__, "CUDA:", th.version.cuda)
print("GPU:", th.cuda.get_device_name(0))

OUT = Path.cwd() / "submission_cuda_outputs" / "cartpole"
for part in ("models", "logs", "results", "figures", "videos"):
    (OUT / part).mkdir(parents=True, exist_ok=True)
print("Output:", OUT)

## 2. Environment

The observation is four numeric state values; the action space is discrete.

In [ ]:
def make_cartpole(log_name=None, render_mode=None):
    def factory():
        env = gym.make("CartPole-v1", render_mode=render_mode)
        return Monitor(env, filename=str(OUT / "logs" / log_name) if log_name else None)
    return DummyVecEnv([factory])

probe = gym.make("CartPole-v1")
print("State:", probe.observation_space, "Actions:", probe.action_space)
probe.close()

## 3. Evaluation and episode counter

Use the same 100 seeds for random, DQN and PPO.

In [ ]:
def evaluate_cartpole(model, episodes=100, random_policy=False, seed_offset=10_000):
    env = gym.make("CartPole-v1")
    rows = []
    try:
        for i in range(episodes):
            seed = SEED + seed_offset + i
            obs, _ = env.reset(seed=seed)
            env.action_space.seed(seed)
            total, length = 0.0, 0
            while True:
                action = env.action_space.sample() if random_policy else int(model.predict(obs, deterministic=True)[0])
                obs, reward, terminated, truncated, _ = env.step(action)
                total += float(reward); length += 1
                if terminated or truncated: break
            rows.append(dict(seed=seed, reward=total, length=length))
    finally:
        env.close()
    return pd.DataFrame(rows)

class EpisodeTarget(BaseCallback):
    def __init__(self, target):
        super().__init__(); self.target = target; self.episodes = 0
    def _on_step(self):
        self.episodes += int(np.asarray(self.locals["dones"]).sum())
        return self.episodes < self.target

## 4. Custom ANN

Both algorithms use this custom SB3 feature extractor.

In [ ]:
class CustomANN(BaseFeaturesExtractor):
    """CartPole's four state values -> 128 features; SB3 adds output heads."""
    def __init__(self, observation_space, features_dim=128):
        super().__init__(observation_space, features_dim)
        inputs = int(np.prod(observation_space.shape))
        self.net = nn.Sequential(
            nn.Flatten(), nn.Linear(inputs, 128), nn.ReLU(),
            nn.Linear(128, 128), nn.ReLU(),
            nn.Linear(128, features_dim), nn.ReLU(),
        )

    def forward(self, observations):
        return self.net(observations.float())

ANN_POLICY = dict(
    features_extractor_class=CustomANN,
    features_extractor_kwargs=dict(features_dim=128),
)
print(CustomANN)

## 5. Train DQN

Run DQN to 10,000 completed episodes.

In [ ]:
CARTPOLE_TARGET_EPISODES = 10_000
dqn_env = make_cartpole(log_name="cartpole_dqn")
dqn = DQN("MlpPolicy", dqn_env, policy_kwargs=ANN_POLICY, device=DEVICE,
          learning_rate=1e-3, buffer_size=100_000, learning_starts=1_000,
          batch_size=64, gamma=0.99, exploration_fraction=0.2,
          exploration_final_eps=0.05, verbose=1, seed=SEED)
print("DQN ANN:", dqn.policy.features_extractor)
print("DQN parameters:", sum(p.numel() for p in dqn.policy.parameters()))
dqn_counter = EpisodeTarget(CARTPOLE_TARGET_EPISODES)
dqn.learn(total_timesteps=10_000_000, callback=dqn_counter)
dqn.save(OUT / "models" / "cartpole_dqn_final")
dqn_training = dict(algorithm="cartpole_dqn", episodes=dqn_counter.episodes,
                    timesteps=dqn.num_timesteps)
dqn_env.close()
display(pd.DataFrame([dqn_training]))

## 6. Train PPO

Run PPO independently to 10,000 completed episodes.

In [ ]:
ppo_env = make_cartpole(log_name="cartpole_ppo")
ppo = PPO("MlpPolicy", ppo_env, policy_kwargs=ANN_POLICY, device=DEVICE,
          learning_rate=3e-4, n_steps=2_048, batch_size=64,
          gamma=0.99, verbose=1, seed=SEED)
print("PPO ANN:", ppo.policy.features_extractor)
print("PPO parameters:", sum(p.numel() for p in ppo.policy.parameters()))
ppo_counter = EpisodeTarget(CARTPOLE_TARGET_EPISODES)
ppo.learn(total_timesteps=10_000_000, callback=ppo_counter)
ppo.save(OUT / "models" / "cartpole_ppo_final")
ppo_training = dict(algorithm="cartpole_ppo", episodes=ppo_counter.episodes,
                    timesteps=ppo.num_timesteps)
ppo_env.close()
training_summary = pd.DataFrame([dqn_training, ppo_training])
training_summary.to_csv(OUT / "results" / "cartpole_training_summary.csv", index=False)
display(training_summary)
assert all(training_summary.episodes >= 1_000)

## 7. Training curves

Training reward and a 100-episode moving average are shown separately.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, name in zip(axes, ("cartpole_dqn", "cartpole_ppo")):
    data = pd.read_csv(OUT / "logs" / f"{name}.monitor.csv", comment="#")
    data["episode"] = np.arange(1, len(data) + 1)
    data["rolling100"] = data.r.rolling(100, min_periods=20).mean()
    ax.plot(data.episode, data.r, alpha=0.12, label="Episode reward")
    ax.plot(data.episode, data.rolling100, label="100-episode mean")
    ax.set(title=f"{name}: {len(data)} episodes", xlabel="Episode", ylabel="Reward")
    ax.grid(alpha=0.2); ax.legend()
fig.tight_layout()
fig.savefig(OUT / "figures" / "cartpole_training.png", dpi=180)
plt.show()

## 8. Independent final evaluation

Reload saved models and compare against the random baseline.

In [ ]:
random_results = evaluate_cartpole(None, random_policy=True)
random_results.to_csv(OUT / "results" / "random_episodes.csv", index=False)
rows = [dict(algorithm="random", train_episodes=0, evaluation_episodes=100,
             mean_return=random_results.reward.mean(), std_return=random_results.reward.std(ddof=1))]
groups = [random_results.reward]
for name, cls in (("cartpole_dqn", DQN), ("cartpole_ppo", PPO)):
    model = cls.load(OUT / "models" / f"{name}_final.zip", device=DEVICE)
    result = evaluate_cartpole(model)
    result.to_csv(OUT / "results" / f"{name}_episodes.csv", index=False)
    train_episodes = int(training_summary.loc[training_summary.algorithm == name, "episodes"].iloc[0])
    rows.append(dict(algorithm=name, train_episodes=train_episodes, evaluation_episodes=len(result),
                     mean_return=result.reward.mean(), std_return=result.reward.std(ddof=1)))
    groups.append(result.reward)
final_results = pd.DataFrame(rows)
final_results.to_csv(OUT / "results" / "cartpole_summary.csv", index=False)
display(final_results)
fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot(groups, tick_labels=["Random", "DQN", "PPO"], showmeans=True)
ax.set(title="CartPole: 100 fixed-seed evaluation episodes", ylabel="Episode reward")
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
fig.savefig(OUT / "figures" / "cartpole_comparison.png", dpi=180)
plt.show()

## 9. MP4 simulations

One complete deterministic episode for each trained algorithm.

In [ ]:
from gymnasium.wrappers import RecordVideo
for name, cls in (("cartpole_dqn", DQN), ("cartpole_ppo", PPO)):
    model = cls.load(OUT / "models" / f"{name}_final.zip", device=DEVICE)
    env = RecordVideo(gym.make("CartPole-v1", render_mode="rgb_array"),
                      video_folder=str(OUT / "videos"), name_prefix=name,
                      episode_trigger=lambda episode: episode == 0, disable_logger=True)
    obs, _ = env.reset(seed=SEED + 40_000)
    total, length = 0.0, 0
    while True:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, _ = env.step(int(action))
        total += float(reward); length += 1
        if terminated or truncated: break
    env.close()
    path = OUT / "videos" / f"{name}-episode-0.mp4"
    print(name, "reward=", total, "length=", length, "seed=", SEED + 40_000, path)
    display(Video(filename=str(path), embed=False))

## 10. Submission checks

Verify the models, results, figures and videos.

In [ ]:
required = [OUT / "results" / "cartpole_summary.csv",
            OUT / "figures" / "cartpole_training.png",
            OUT / "figures" / "cartpole_comparison.png"]
for name in ("cartpole_dqn", "cartpole_ppo"):
    required += [OUT / "models" / f"{name}_final.zip",
                 OUT / "videos" / f"{name}-episode-0.mp4",
                 OUT / "results" / f"{name}_episodes.csv"]
missing = [str(p) for p in required if not p.exists()]
assert not missing, f"Missing artifacts: {missing}"
assert all(training_summary.episodes >= 1_000)
print("All CartPole artifacts exist. Evaluation table above is the report source.")